# AI Robotics - Projectile Tracking System
## Object Detection | PID Gimbal Control | Laser Alignment Simulation

Simulation of an AI-powered tracking system with gimbal control.
Ready for real hardware (servo motors, camera, laser pointer) deployment.

---

In [ ]:
# CELL 1: Setup
import subprocess, sys
for p in ['numpy','matplotlib','scipy','opencv-python']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from scipy.integrate import odeint
from pathlib import Path
np.random.seed(42)
OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
print('Robotics Sim Ready!')

In [ ]:
# CELL 2: Projectile Physics Simulation
def projectile_trajectory(v0, angle_deg, dt=0.01, g=9.81):
    """Simulate 2D projectile motion with air resistance."""
    angle = np.radians(angle_deg)
    vx, vy = v0 * np.cos(angle), v0 * np.sin(angle)
    x, y = [0], [0]
    drag = 0.01  # Air resistance coefficient
    
    while y[-1] >= 0 or len(x) < 3:
        speed = np.sqrt(vx**2 + vy**2)
        ax = -drag * speed * vx
        ay = -g - drag * speed * vy
        vx += ax * dt; vy += ay * dt
        x.append(x[-1] + vx * dt); y.append(y[-1] + vy * dt)
        if y[-1] < 0 and len(x) > 10: break
    
    return np.array(x), np.array(y)

# Generate multiple trajectories
fig, ax = plt.subplots(figsize=(12, 6))
for angle in [30, 45, 60, 75]:
    x, y = projectile_trajectory(v0=30, angle_deg=angle)
    ax.plot(x, y, linewidth=2, label=f'{angle} degrees')

ax.set_title('Projectile Trajectories (with Air Resistance)')
ax.set_xlabel('Distance (m)'); ax.set_ylabel('Height (m)')
ax.legend(); ax.set_ylim(bottom=0); ax.grid(True, alpha=0.3)
plt.savefig(OUTPUT_DIR / 'trajectories.png', dpi=150); plt.show()

In [ ]:
# CELL 3: PID Controller for Gimbal Tracking
class PIDController:
    """PID controller for servo gimbal aiming.
    
    P = Proportional (react to current error)
    I = Integral (correct accumulated error over time)
    D = Derivative (dampen oscillations)
    """
    def __init__(self, kp=2.0, ki=0.1, kd=0.5, dt=0.01):
        self.kp, self.ki, self.kd, self.dt = kp, ki, kd, dt
        self.integral = 0; self.prev_error = 0
    
    def update(self, error):
        self.integral += error * self.dt
        self.integral = np.clip(self.integral, -10, 10)  # Anti-windup
        derivative = (error - self.prev_error) / self.dt
        output = self.kp * error + self.ki * self.integral + self.kd * derivative
        self.prev_error = error
        return output
    
    def reset(self):
        self.integral = 0; self.prev_error = 0

# Simulate tracking a projectile
target_x, target_y = projectile_trajectory(v0=25, angle_deg=45, dt=0.02)
n_steps = len(target_x)
dt = 0.02

# Gimbal state: pan (horizontal) and tilt (vertical) angles
pid_pan = PIDController(kp=3.0, ki=0.2, kd=0.8, dt=dt)
pid_tilt = PIDController(kp=3.0, ki=0.2, kd=0.8, dt=dt)

gimbal_origin = np.array([0, 0])
gimbal_pan = np.zeros(n_steps)
gimbal_tilt = np.zeros(n_steps)
tracking_error = np.zeros(n_steps)

for i in range(1, n_steps):
    # Target angles from gimbal
    dx = target_x[i] - gimbal_origin[0]
    dy = target_y[i] - gimbal_origin[1]
    target_pan = np.degrees(np.arctan2(dx, 1))  # Simplified 2D
    target_tilt = np.degrees(np.arctan2(dy, dx + 0.001))
    
    # PID control
    pan_error = target_pan - gimbal_pan[i-1]
    tilt_error = target_tilt - gimbal_tilt[i-1]
    
    gimbal_pan[i] = gimbal_pan[i-1] + pid_pan.update(pan_error) * dt
    gimbal_tilt[i] = gimbal_tilt[i-1] + pid_tilt.update(tilt_error) * dt
    
    # Tracking error (angular)
    tracking_error[i] = np.sqrt(pan_error**2 + tilt_error**2)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
t = np.arange(n_steps) * dt

axes[0,0].plot(target_x, target_y, 'r-', linewidth=2, label='Projectile')
axes[0,0].plot(0, 0, 'g^', markersize=15, label='Gimbal Mount')
axes[0,0].set_title('Target Trajectory'); axes[0,0].legend()

axes[0,1].plot(t, tracking_error, 'b-', linewidth=1)
axes[0,1].set_title('Tracking Error Over Time'); axes[0,1].set_ylabel('Error (degrees)')

axes[1,0].plot(t, gimbal_pan, label='Pan'); axes[1,0].plot(t, gimbal_tilt, label='Tilt')
axes[1,0].set_title('Gimbal Angles'); axes[1,0].legend()

axes[1,1].bar(['Mean Error', 'Max Error', 'Settling Time'], 
    [tracking_error[10:].mean(), tracking_error.max(), t[np.argmax(tracking_error < 1)] if np.any(tracking_error < 1) else t[-1]],
    color=['green', 'red', 'blue'])
axes[1,1].set_title('Performance Metrics')

plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'pid_tracking.png', dpi=150); plt.show()
print(f"Mean tracking error: {tracking_error[10:].mean():.2f} degrees")
print("\nP9 AI Robotics Projectile Tracking - COMPLETE!")